#Lab.13 / IBM3202 – Molecular Docking of Multiple Ligands (Virtual Screening)

###Theoretical aspects

As commented on [Lab.06](https://colab.research.google.com/github/pb3lab/ibm3202/blob/master/tutorials/lab06_docking.ipynb), **molecular docking** explores the potential binding poses of small molecules on the **binding site** of a target protein for which an experimentally determined structure is available. 

When used for drug discovery, typically one would like to perform the docking of many molecules against a given target protein to identify those that are most likely to bind to it.

This strategy is called **virtual screening** and will be explored in this tutorial. We will also explore the possibility of using flexible side chains in the protein.

#Part 0 – Downloading and Installing the required software

Before we start, you must first **remember to start the hosted runtime in Google Colab**.

Then, we must install several pieces of software to perform our task. Namely:
- **py3Dmol** for visualization of the protein structure and setting up the search grid.
- **biopython** for downloading and manipulating protein structures
- **miniconda**, a free minimal installer of **conda** for software package and environment management.
- **ChEMBL web resource client** for downloading the ligands
- **OpenBabel** for parameterization of our ligand(s).
- **MGLtools** for parameterization of our target protein using Gasteiger charges.
- **pdb2pqr** for parameterization of our protein using the AMBER ff99 forcefield.
- **Autodock Vina** for the docking process
- **QuickVina** for faster docking processes

After several tests, the following installation instructions are the best way of setting up **Google Colab** for this laboratory session.

1. We will first install py3Dmol, biopython and ChEMBL as follows:

In [ ]:
#Installing py3Dmol using pip
!pip install py3Dmol
#Installing biopython using pip
!pip install biopython
#We will also install kora for using RDkit
!pip install kora
#Finally, we will install ChEMBL
!pip install chembl_webresource_client

     |████████████████████████████████| 2.3 MB 2.8 MB/s 
     |████████████████████████████████| 57 kB 2.5 MB/s 
     |████████████████████████████████| 56 kB 3.9 MB/s 
     |████████████████████████████████| 55 kB 2.2 MB/s 
     |████████████████████████████████| 636 kB 8.7 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13
  Attempting uninstall: itsdangerous
    Found existing installation: itsdangerous 1.1.0
    Uninstalling itsdangerous-1.1.0:
      Successfully uninstalled itsdangerous-1.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
flask 1.1.4 requires itsdangerous<2.0,>=0.24, but you have itsdangerous 2.0.1 which is incompatible.


In [ ]:
#Importing py3Dmol for safety
import py3Dmol

2. Then, we will install PDB2PQR using apt-get as follows:

In [ ]:
#Installing pdb2pqr using apt-get
!apt-get install -y pdb2pqr

Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following additional packages will be installed:
  python-decorator python-networkx python-pkg-resources python-yaml
Suggested packages:
  apbs python-matplotlib python-pydotplus python-scipy python-pygraphviz
  | python-pydot python-setuptools
The following NEW packages will be installed:
  pdb2pqr python-decorator python-networkx python-pkg-resources python-yaml
0 upgraded, 5 newly installed, 0 to remove and 37 not upgraded.
Need to get 1,431 kB of archives.
After this operation, 7,952 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu bionic/main amd64 python-decorator all 4.1.2-1 [9,300 B]
Get:2 http://archive.ubuntu.com/ubuntu bionic-updates/main amd64 python-networkx all 1.11-1ubuntu3 [804 kB]
Get:3 http://archive.ubuntu.com/ubuntu bionic/universe amd64 pdb2pqr amd64 2.1.1+dfsg-2 [375 kB]
Get:4 http://archive.ubuntu.com/ubuntu bionic/main amd64 python-

In [ ]:
#Checking that pdb2pqr was properly installed
!pdb2pqr -h

3. And then we will install conda to be able to install MGLtools and OpenBabel

In [ ]:
#Install conda using the new conda-colab library
!pip install -q condacolab
import condacolab
condacolab.install_miniconda()

#Install MGLtools and OpenBabel from
#the bioconda repository
!conda install -c conda-forge -c bioconda mgltools openbabel zlib --yes

⏬ Downloading https://repo.anaconda.com/miniconda/Miniconda3-py37_4.9.2-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:32
🔁 Restarting kernel...
Solving environment: | / - \ | / - \ | / - \ | / done


==> WARNING: A newer version of conda exists. <==
  current version: 4.9.2
  latest version: 4.10.3

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - mgltools
    - openbabel
    - zlib


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2021.5.30  |       ha878542_0         136 KB  conda-forge
    cairo-1.16.0               |    h3fc0475_1005         1.5 MB  conda-forge
    certifi-2021.5.30          |   py37h89c1867_0         141 KB  conda-forge
    conda-4.10.3             

4. Finally, we will download the Autodock Vina program from the Scripps website and make an alias to use it during this session

In [ ]:
#Download and extract Autodock Vina from SCRIPPS
#Then, we set up an alias for vina to be treated as a native binary
%%bash
wget http://vina.scripps.edu/download/autodock_vina_1_1_2_linux_x86.tgz
tar xzvf autodock_vina_1_1_2_linux_x86.tgz

autodock_vina_1_1_2_linux_x86/
autodock_vina_1_1_2_linux_x86/LICENSE
autodock_vina_1_1_2_linux_x86/bin/
autodock_vina_1_1_2_linux_x86/bin/vina
autodock_vina_1_1_2_linux_x86/bin/vina_split


--2021-10-07 16:36:31--  http://vina.scripps.edu/download/autodock_vina_1_1_2_linux_x86.tgz
Resolving vina.scripps.edu (vina.scripps.edu)... 137.131.108.109
Connecting to vina.scripps.edu (vina.scripps.edu)|137.131.108.109|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1238242 (1.2M) [application/x-gzip]
Saving to: ‘autodock_vina_1_1_2_linux_x86.tgz’

     0K .......... .......... .......... .......... ..........  4% 96.5K 12s
    50K .......... .......... .......... .......... ..........  8%  384K 7s
   100K .......... .......... .......... .......... .......... 12%  385K 5s
   150K .......... .......... .......... .......... .......... 16%  387K 5s
   200K .......... .......... .......... .......... .......... 20%  389K 4s
   250K .......... .......... .......... .......... .......... 24% 57.4M 3s
   300K .......... .......... .......... .......... .......... 28%  386K 3s
   350K .......... .......... .......... .......... .......... 33% 74.7M 2s
   400K ...

In [ ]:
alias vina /content/autodock_vina_1_1_2_linux_x86/bin/vina

**⚠️WARNING:** We will be soon updated to the new Autodock Vina v1.2.2, which can be found [here](https://github.com/ccsb-scripps)

5. Alternatively, we can download QuickVina 2, a redesigned version of AutoDock Vina that is 20 times faster and equally accurate.


In [ ]:
%%bash
#Downloading Quick Vina and enabling its execution
git clone https://github.com/QVina/qvina
chmod +x /content/qvina/bin/qvina2.1

Cloning into 'qvina'...


In [ ]:
alias qvina2 /content/qvina/bin/qvina2.1

Once these software installation processes are completed, we are ready to perform our experiments

#Part 1 – Preparing the Receptor for Virtual Screening with rigid or flexible side chains

1. The first step in a molecular docking procedure is to have a structure of a given target protein. While in some cases a high-quality comparative model will be used, most cases start with an experimentally (X-ray, NMR, cryoEM) solved three-dimensional structure. 

  In such cases, a given target protein structure is downloaded from the **Protein Data Bank (PDB)** (https://www.rcsb.org/pdb) using a given accession ID. For example, the PET hydrolase solved by our lab has the accession ID 6ANE.

  For this tutorial, we will again use the crystal structure of the HIV-2 protease as we did for [Lab.06](https://colab.research.google.com/github/pb3lab/ibm3202/blob/master/tutorials/lab06_docking.ipynb), which was deposited in the PDB with the accession ID 1HSG.
  
  A difference with the aforementioned tutorial is that we will use **biopython** for retrieving our PDB file, and we will also do some post-processing, namely eliminating alternative residue conformations (i.e. we are only keeping one conformation when the occupancies < 1.0) and all non-protein atoms (waters, ions, ligands, etc)

In [8]:
#Let's make sure we are on the main directory
import os
os.chdir('/content/')

#Let's make a folder first. We need to import the os and path library
from pathlib import Path 

#Then, we define the path of the folder we want to create.
#Notice that the HOME folder for a hosted runtime in colab is /content/
singlepath = Path("/content/VSdocking/")

#Now, we create the folder using the os.mkdir() command
#The if conditional is just to check whether the folder already exists
#In which case, python returns an error
if os.path.exists(singlepath):
  print("Virtual screening path already exists")
if not os.path.exists(singlepath):
  os.mkdir(singlepath)
  print("Virtual screening path was succesfully created")

#Now we will change to the new folder
os.chdir(singlepath)

#Importing your PDB file using biopython
from Bio.PDB import *

#Here we set out PDB ID
pdbid = ['1hsg']
pdbl = PDBList()
for s in pdbid:
  pdbl.retrieve_pdb_file(s, pdir='.', file_format ="pdb", overwrite=True)
  os.rename("pdb"+s+".ent", s+".pdb")

Virtual screening path already exists


In [9]:
#Here we set up a parser for our PDB
pdb = PDBParser().get_structure('X', '1hsg.pdb')
io=PDBIO()

#And here we set the residue conformation we want to keep
keepAltID = "A"

class KeepOneConfOnly(Select):  # Inherit methods from Select class
    def accept_atom(self, atom):
        if (not atom.is_disordered()) or atom.get_altloc() == keepAltID:
            atom.set_altloc(" ")  # Eliminate alt location ID before output.
            return True
        else:  # Alt location was not one to be output.
            return False
        # end of accept_atom()

#This will keep only conformation for each residue
io.set_structure(pdb)
io.save("1hsg_pass1.pdb", select=KeepOneConfOnly())
print("Your PDB was processed. Alternative side chain conformations removed!")

#Here we set up parser for our second PDB
pdb = PDBParser().get_structure('X', "1hsg_pass1.pdb")
io = PDBIO()

#Here we create a HETATM selector to removing all HETATM (water, ligands, etc)
class NonHetSelect(Select):
    def accept_residue(self, residue):
        return 1 if residue.id[0] == " " else 0

io.set_structure(pdb)
io.save("1hsg_prot.pdb", NonHetSelect())
print("Your PDB was processed. Only the protein heavy atoms have been kept!")

Your PDB was processed. Alternative side chain conformations removed!
Your PDB was processed. Only the protein heavy atoms have been kept!


/usr/local/lib/python3.7/dist-packages/Bio/PDB/StructureBuilder.py:92: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 1924.
  PDBConstructionWarning,
/usr/local/lib/python3.7/dist-packages/Bio/PDB/StructureBuilder.py:92: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 1968.
  PDBConstructionWarning,


2. For AutoDock to perform a molecular docking experiment, the protein target must contain information about the partial charges of each atom and atom types that are compatible with AutoDock. Such format is referred to as **PDBQT**, a modification of the PDB format that also includes **charges (q)** and **AutoDock-specific atom types (t)** in two extra columns at the end of the now PDBQT file.

  Lastly, the protein target must contain **all polar hydrogens**. Most protein structures have no hydrogens included, meaning that we must add them. We will employ first strategy 2a for parameterization and, in case it is not optimal, we will employ strategy 2b.

2. a) We will now add the polar hydrogens of our protein and parameterize it with **Gasteiger** charges and atom types using **MGLtools** (this is the canonical option for the majority of AutoDock users)

In [ ]:
#Parameterizing and adding Gasteiger charges into our protein using MGLtools
!pythonsh /usr/local/bin/prepare_receptor4.py -r $singlepath/*.pdb -o $singlepath/BiP.pdbqt -A hydrogens -U nphs_lps -v

set verbose to  True
read  /content/dockingBiPMg/robetta_Trmodels_52159_3_plusMG.pdb
setting up RPO with mode= automatic and outputfilename=  /content/dockingBiPMg/BiP.pdbqt
charges_to_add= gasteiger
delete_single_nonstd_residues= None
Unable to assign HAD type to atom Mg
Unable to assign valence to atom robetta_Trmodels_52159_3_plusMG:A: MG608:MG type = Mg
Unable to assign MAP type to atom Mg
Sorry, there are no Gasteiger parameters available for atom robetta_Trmodels_52159_3_plusMG:A: MG608:MG


2. b) Alternatively, we will add the polar hydrogens of our protein and parameterize it based on the pKa of each aminoacid at pH 7.4 with the **AMBER99ff** force field using **pdb2pqr**, followed by deletion of non-polar hydrogens and conversion into **PDBQT** file using **MGLtools**.

  In this case, pdb2pqr generates an intermediate **PQR** file, a modification of the PDB format which allows users to add charge and radius parameters to existing PDB data. This information is then unaltered during the use of **MGLtools**.

In [ ]:
#First, using pdb2pqr to parameterize our receptor with AMBER99ff, maintaining
#the chain IDs and setting up the receptor at a pH of 7.4
!pdb2pqr --ff=amber --chain --with-ph=7.4 --verbose $singlepath/*.pdb $singlepath/BiP.pqr

#Then, convert the .pqr file into a .pdbqt file while deleting non-polar
#hydrogens but without changing the AMBER parameters added to the protein
!pythonsh /usr/local/bin/prepare_receptor4.py -r $singlepath/BiP.pqr -o $singlepath/BiP.pdbqt -C -U nphs_lps -v

#Part 2 – Downloading and Preparing the Ligands for AutoDock

1. We will first start by creating a different folder for each ligand, in which we will store our ligands separately for molecular docking.

In [ ]:
#Let's make a folder first. We need to import the os and path library
import os
from pathlib import Path

#We will first create a path for all ligands that we will use in this tutorial
#Notice that the HOME folder for a hosted runtime in colab is /content/
#Ligands = ATP, AMP-PNP, ATPgammaS, ADP

ligands = ['CHEMBL14249', 'CHEMBL2220361', 'CHEMBL131890', 'CHEMBL14830']

for l in ligands:
  ligandpath = Path("/content/" + l)
  if os.path.exists(ligandpath):
    print("ligand path already exists")
  if not os.path.exists(ligandpath):
    os.mkdir(ligandpath)
  print("ligand path " + str(ligandpath) + " was succesfully created")

ligand path /content/CHEMBL14249 was succesfully created
ligand path /content/CHEMBL2220361 was succesfully created
ligand path /content/CHEMBL131890 was succesfully created
ligand path /content/CHEMBL14830 was succesfully created


2. Now, we will download ATP (CHEMBL14249), AMP-PNP (CHEMBL2220361), ATP$\gamma$S (CHEMBL131890) and ADP (CHEMBL14830) from the **ChEMBL** database (https://www.ebi.ac.uk/chembl/). This is a comprehensive, freely accessible, online database containing information on different compounds, drugs and drug targets. 

  We will download this ligand in SMILES format to continue with its preparation for molecular docking

In [ ]:
#Downloading Ligands from ChEMBL database
from chembl_webresource_client.new_client import new_client
molecule = new_client.molecule
ligands = ['CHEMBL14249', 'CHEMBL2220361', 'CHEMBL131890', 'CHEMBL14830']
for l in ligands:
  ligandpath = Path("/content/" + l)
  with open(ligandpath / "ligand.smiles","w") as f:
    m1 = molecule.get(l)
    f.write(m1['molecule_structures']['canonical_smiles'])

3. **Let's take a look at the SMILES of each molecule**

In [ ]:
#Print the SMILES of all ligands
for l in ligands:
  ligandpath = Path("/content/" + l)
  print(l)
  print((ligandpath / "ligand.smiles").read_text())

CHEMBL14249
Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O)OP(=O)(O)O)[C@@H](O)[C@H]1O
CHEMBL2220361
N=P(O)(OP(=O)(O)O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1O
CHEMBL131890
Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O)OP(O)(O)=S)[C@@H](O)[C@H]1O
CHEMBL14830
Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O)O)[C@@H](O)[C@H]1O


In [ ]:
#Use the following viewer to load your SMILES as a 3D molecule
import py3Dmol
import kora.install.rdkit
from rdkit import Chem
from rdkit.Chem import AllChem

def MolTo3DView(mol, size=(300, 300), style="stick", surface=False, opacity=0.5):
    assert style in ('line', 'stick', 'sphere', 'carton')
    mblock = Chem.MolToMolBlock(mol)
    viewer = py3Dmol.view()
    viewer.addModel(mblock, 'mol')
    viewer.setStyle({style:{}})
    if surface:
        viewer.addSurface(py3Dmol.SAS, {'opacity': opacity})
    viewer.zoomTo()
    return viewer

from ipywidgets import interact,fixed,IntSlider
import ipywidgets

def smi2conf(smiles):
    '''Convert SMILES to rdkit.Mol with 3D coordinates'''
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol)
        AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
        return mol
    else:
        return None

@interact
def smi2viewer(smi='CC=O'):
    try:
        conf = smi2conf(smi)
        return MolTo3DView(conf).show()
    except:
        return None

interactive(children=(Text(value='CC=O', description='smi'), Output()), _dom_classes=('widget-interact',))

4. Now, we will take this SMILES format and use it to construct and parameterize a three-dimensional structure of all ligands in **PDBQT** format for its use in molecular docking. As with the receptor, we also have different options to prepare our ligand for molecular docking:

4. a) Use the program **babel** to convert the SMILES into a **MOL2** file without any extra work (such as searching for best conformers) except for setting the protonation state to pH 7.4, and then use **MGLtools** to parameterize the ligand using **Gasteiger** partial charges (this is the canonical option for the majority of AutoDock users).

  Please note that we are generating a ligand in which **all torsions are active** during the docking procedure.

In [ ]:
#Change to the ligand directory
for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
#Converting all SMILES into a 3D PDB format
  !obabel ligand.smiles -O ligand.mol2 --gen3d best -p 7.4 --canonical
#Parameterizing and adding Gasteiger charges into our protein using MGLtools
#Adding -z leads to a rigid ligand without any torsions
  !pythonsh /usr/local/bin/prepare_ligand4.py -l ligand.mol2 -o ligand.pdbqt -U nphs_lps -v
#NOTE: for some reason, MGLtools does not recognize the ligand when inside a different folder
#Here we are deleting the temporary PDB file required for generating the PDBQT file
  os.remove("ligand.mol2")

#Go back to the home directory
os.chdir("/content/")

1 molecule converted
set verbose to  True
read  ligand.mol2
setting up LPO with mode= automatic and outputfilename=  ligand.pdbqt
and check_for_fragments= False
and bonds_to_inactivate= 
returning  0
No change in atomic coordinates
1 molecule converted
set verbose to  True
read  ligand.mol2
setting up LPO with mode= automatic and outputfilename=  ligand.pdbqt
and check_for_fragments= False
and bonds_to_inactivate= 
returning  0
No change in atomic coordinates
1 molecule converted
set verbose to  True
read  ligand.mol2
setting up LPO with mode= automatic and outputfilename=  ligand.pdbqt
and check_for_fragments= False
and bonds_to_inactivate= 
returning  0
No change in atomic coordinates
1 molecule converted
set verbose to  True
read  ligand.mol2
setting up LPO with mode= automatic and outputfilename=  ligand.pdbqt
and check_for_fragments= False
and bonds_to_inactivate= 
returning  0
No change in atomic coordinates


4. b) Use the program **babel** to  convert the SMILES into a 3D **MOL2** file while simultaneously performing and energy minimization using the Generalized Amber Force Field (**GAFF**). Then, use **MGLtools** to parameterize the ligand using **Gasteiger** partial charges.

  Please note that we are generating a ligand in which **all torsions are active** during the docking procedure.

In [ ]:
#Change to the ligand directory
for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
#Converting ligands from SMILES into a 3D MOL2 format and perform an energy minimization of the conformer using the GAFF forcefield
#Then, prepare ligands for docking using the Autodock script
  !obabel ligand.smiles -O ligand.mol2 --gen3d --best --canonical --minimize --ff GAFF --steps 10000 --sd
  !pythonsh /usr/local/bin/prepare_ligand4.py -l ligand.mol2 -o ligand.pdbqt -U nphs_lps -v
#NOTE: for some reason, MGLtools does not recognize the ligand when inside a different folder
#Here we are deleting the temporary PDB file required for generating the PDBQT file
  os.remove("ligand.mol2")

#Go back to the home directory
os.chdir("/content/")

4. c) Use the program **babel** to  convert the SMILES into a 3D **MOL2** file while simultaneously performing a weighted rotor search for the lowest energy conformer using the Generalized Amber Force Field (**GAFF**). Then, use **MGLtools** to parameterize the ligand using **Gasteiger** partial charges.

  Please note that we are generating a ligand in which **all torsions are active** during the docking procedure.

In [ ]:
#Change to the ligand directory
for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
#Converting ligands from SMILES into a 3D MOL2 format and perform a weighted rotor search for lowest energy conformer
#Then, prepare ligands for docking using the Autodock script
  !obabel ligand.smiles -O ligand.mol2 --gen3d --best --canonical --conformers --weighted --nconf 50 --ff GAFF
  !pythonsh /usr/local/bin/prepare_ligand4.py -l ligand.mol2 -o ligand.pdbqt -U nphs_lps -v
#NOTE: for some reason, MGLtools does not recognize the ligand when inside a different folder
#Here we are deleting the temporary PDB file required for generating the PDBQT file
  os.remove("ligand.mol2")

#Go back to the home directory
os.chdir("/content/")

**You are all set with your ligand!** Now, we move onto setting up the molecular docking experiment

#Part 3 – Setting up and Performing Molecular Docking with AutoDock

1. As explained in the lectures, it is necessary to define the search space for molecular docking on a given target protein through the use of a **grid box**. This grid box is usually centered within the binding, active or allosteric site of the target protein and its size will be sufficiently large such that **all binding residues are placed inside the grid box**.

  Here, we will make use of **py3Dmol** to visually inspect the protein structure in cartoon representation and to draw a grid box. The position and size of the grid box will be defined by the coordinates of its centroid and by its dimensions in x, y and z.

  To better guide the search for the optimal dimensions and coordinates of the grid box, we will also show the residues Val32, Ile47 and Val82 of HIV-2 protease.

  The script that defines the visualizer, which we called **ViewProtGrid**, is first loaded into **Colab** with the following lines of code

In [ ]:
#These definitions will enable loading our protein and then
#drawing a box with a given size and centroid on the cartesian space
#This box will enable us to set up the system coordinates for the simulation
#
#First, we define the grid box
def definegrid(object,bxi,byi,bzi,bxf,byf,bzf):
  object.addBox({'center':{'x':bxi,'y':byi,'z':bzi},'dimensions': {'w':bxf,'h':byf,'d':bzf},'color':'blue','opacity': 0.6})

#Next, we define how the protein will be shown in py3Dmol
#Note that we are also adding a style representation for active site residues
def viewprot(object,prot_PDBfile,resids):
  mol1 = open(prot_PDBfile, 'r').read()
  object.addModel(mol1,'pdb')
  object.setStyle({'cartoon': {'color':'spectrum'}})
  object.addStyle({'resi':resids},{'stick':{'colorscheme':'greenCarbon'}})

#Lastly, we combine the box grid and protein into a single viewer
def viewprotgrid(prot_PDBfile,resids,bxi,byi,bzi,bxf=10,byf=10,bzf=10):
  mol_view = py3Dmol.view(800,400) # view(800,400,viewergrid=(1,2))
  definegrid(mol_view,bxi,byi,bzi,bxf,byf,bzf)
  viewprot(mol_view,prot_PDBfile,resids)
  mol_view.setBackgroundColor('0xffffff')
  #mol_view.rotate(90, {'x':0,'y':1,'z':0},viewer=(0,1));
  mol_view.zoomTo()
  mol_view.show() 


2. Now, we will use our ViewProtGrid to visualize the protein, binding site residues and a grid box of variable size and position that we can manipulate using a slider through *ipywidgets*. You have to edit this viewer by indicating the location of the PDB file in the *prot_PDBfile* variable (e.g. singlepath/'1HSG_prot.pdb') and the residues that you want to show from the PDB in the *resids* variable.


Examples of how to use the *protein_PDBfile* variable
>prot_PDBfile = ['1HSG_prot.pdb'] (if the PDB file is in the current path)

>prot_PDBfile = [singlepath/'1HSG_prot.pdb'] (if the PDB file is in a path defined as singlepath)


Examples of how to use the *resids* variable

>resids = [82] shows a single residue in position 82)

>resids = ['82,83,84'] shows residues 82, 83 or 84 separately, which you can select in the viewer

>resids = ['(82,83,84)'] shows residue 82, 83 and 84 in the same visualization

>resids = ['82-84'] shows residue range 82-84 in the same visualization

**NOTE:** This code fails when attempting to show two non-consecutive residues in the same visualization.


In [ ]:
from ipywidgets import interact,fixed,IntSlider
import ipywidgets
interact(viewprotgrid,
         prot_PDBfile = [singlepath/'robetta_Trmodels_52159_3_plusMG.pdb'],
         resids = [(38,72,176,234,268,271,272,275)],
         bxi=ipywidgets.IntSlider(min=-100,max=100, step=1),
         byi=ipywidgets.IntSlider(min=-100,max=100, step=1),
         bzi=ipywidgets.IntSlider(min=-100,max=100, step=1),
         bxf=ipywidgets.IntSlider(min=0,max=30, step=1),
         byf=ipywidgets.IntSlider(min=0,max=30, step=1),
         bzf=ipywidgets.IntSlider(min=0,max=30, step=1))

interactive(children=(Dropdown(description='prot_PDBfile', options=(PosixPath('/content/dockingBiPMg/robetta_T…

<function __main__.viewprotgrid>

3. Now, we will generate a configuration file for **Autodock**. As expected, the configuration file contains information about the target protein and ligand, as well as the position and dimensions of the grid box that defines the search space.

  After careful inspection of an adequate box grid, the origin of the box for our comparative model [-17, 7, -33] and its size is [24, 24, 24].

  For defining the grid box, you will use the box origin and size coordinates that you defined manually in the previous step.

  The following is an example file of a standard **Autodock configuration file**, including all possible variables that can be edited:


```
#CONFIGURATION FILE

#INPUT OPTIONS 
receptor = [target protein pdbqt file]
ligand = [ligand pdbqt file]
flex = [flexible residues in receptor in pdbqt format] 

#SEARCH SPACE CONFIGURATIONS 
#Center of the box (coordinates x, y and z 
center_x = [value] 
center_y = [value]
center_z = [value]
#Size of the box (dimensions in x, y and z) 
size_x = [value]
size_y = [value]
size_z = [value]

#OUTPUT OPTIONS 
#out = [output pdbqt file for all conformations]
#log = [output log file for binding energies]

#OTHER OPTIONS 
cpu = [value] # more cpus reduces the computation time
exhaustiveness = [value] # search time for finding the global minimum, default is 8
num_modes = [value] # maximum number of binding modes to generate, default is 9
energy_range = [value] # maximum energy difference between the best binding mode and the worst one displayed (kcal/mol), default is 3
seed = [value] # explicit random seed, not required
```

The following script will create this file for our docking procedure. **You will need to add the position and dimensions of your grid box**


In [ ]:
#Module for copying files
from shutil import copyfile

for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
  copyfile(singlepath/"BiP.pdbqt", ligandpath/"BiP.pdbqt")
  with open("config_singledock","w") as f:
    f.write("#CONFIGURATION FILE (options not used are commented) \n")
    f.write("\n")
    f.write("#INPUT OPTIONS \n")
    f.write("receptor = BiP.pdbqt \n")
    f.write("ligand = ligand.pdbqt \n")
    f.write("#flex = [flexible residues in receptor in pdbqt format] \n")
    f.write("#SEARCH SPACE CONFIGURATIONS \n")
    f.write("#Center of the box (values bxi, byi and bzi) \n")
    f.write("center_x = -17 \n")
    f.write("center_y = 3 \n")
    f.write("center_z = -33 \n")
    f.write("#Size of the box (values bxf, byf and bzf) \n")
    f.write("size_x = 24 \n")
    f.write("size_y = 20 \n")
    f.write("size_z = 24 \n")
    f.write("#OUTPUT OPTIONS \n")
    f.write("#out = \n")
    f.write("#log = \n")
    f.write("\n")
    f.write("#OTHER OPTIONS \n")
    f.write("#cpu =  \n")
    f.write("#exhaustiveness = \n")
    f.write("#num_modes = \n")
    f.write("#energy_range = \n")
    f.write("#seed = ")

os.chdir("/content/")

4. Lastly, we will enter into the folder that we created for the docking experiment and **perform our first molecular docking with Autodock**.

  Once you execute the lines of code shown below, Autodock will show you a progress bar (if running as expected). **This simulation should not take longer than 10 min**.
  
  Note that we are defining the filenames of the output and log file outside the configuration file.

In [ ]:
#Changing directory to each ligand folder for docking
for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
#Executing AutoDock Vina with our configuration file
  %vina --config config_singledock --out output.pdbqt --log log.txt
#Exiting the execution directory
os.chdir("/content/")

#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, Journal of Computational Chemistry 31 (2010)  #
# 455-461                                                       #
#                                                               #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see http://vina.scripps.edu for more information.      #
#################################################################

Detected 2 CPUs
Reading input ... done.
Setting up the scoring function ... done.
Analyzing the binding site ... done.
Using random seed: -16

5. We will split the different docking poses generated as a result of the molecular docking simulation into separate PDB files using **babel**, starting with file numbered as 1 corresponding to the lowest-energy pose.

In [ ]:
#We need to convert our Autodock Vina results from pdbqt into pdb
#For this, we use babel
#Change to the ligand directory
for l in ligands:
  ligandpath = Path("/content/" + l)
  os.chdir(ligandpath)
#Using babel to split the configurations
  !obabel -ipdbqt output.pdbqt -opdb -O output.pdb -m
#Go back to the home directory
os.chdir("/content/")

9 molecules converted
9 files output. The first is output1.pdb
8 molecules converted
8 files output. The first is output1.pdb
9 molecules converted
9 files output. The first is output1.pdb
9 molecules converted
9 files output. The first is output1.pdb


6. Finally, we create another visualizer (**ViewDocking**) to load our protein and any docking pose of our choice.

In [ ]:
#We finally create a visualization of the protein as cartoon,
#the lowest-energy docking pose with its carbons in green
def viewdocking(protein_name,ligand_name):
  mview = py3Dmol.view(800, 400)  
  mol1 = open(protein_name, 'r').read()
  mol2 = open(ligand_name, 'r').read()
  mview.addModel(mol1,'pdb')
  mview.setStyle({'cartoon': {'color':'spectrum'}})
  mview.addStyle({'resn':'MG'},{'sphere':{'color':'yellow'}})
  mview.addModel(mol2,'pdb')
  mview.setStyle({'model':1},{'stick':{'colorscheme':'greenCarbon'}})
  mview.setBackgroundColor('0xffffff')
  mview.zoomTo()
  mview.show()

7. The ViewDocking visualizer can then be used as indicated below.

In [ ]:
#View docking results
#viewdocking('protein_file','docked_ligand_file','exp_ligand_file')
viewdocking('dockingBiPMg/robetta_Trmodels_52159_3_plusMG.pdb','CHEMBL14830/output1.pdb')

You appear to be running in JupyterLab (or JavaScript failed to load for some other reason). You need to install the 3dmol extension: 
 jupyter labextension install jupyterlab_3dmol

It is **recommended** to visualize these docking results in a standalone software (e.g. VMD, PyMOL, Chimera, etc). Thus, we will generate compressed ZIP files for downloading these results. You can then right-click on the generated ZIP files and select *Download* in the drop-down menu.
 

In [ ]:
%%bash
zip -r dockingBiPMg.zip dockingBiPMg 
for n in CHEMBL*
do
echo $n
zip -r $n.zip $n
done

  adding: dockingBiPMg/ (stored 0%)
  adding: dockingBiPMg/robetta_Trmodels_52159_3_plusMG.pdb (deflated 78%)
  adding: dockingBiPMg/BiP.pdbqt (deflated 76%)
CHEMBL131890
  adding: CHEMBL131890/ (stored 0%)
  adding: CHEMBL131890/log.txt (deflated 63%)
  adding: CHEMBL131890/output7.pdb (deflated 81%)
  adding: CHEMBL131890/output6.pdb (deflated 81%)
  adding: CHEMBL131890/ligand.smiles (deflated 34%)
  adding: CHEMBL131890/config_singledock (deflated 40%)
  adding: CHEMBL131890/output2.pdb (deflated 81%)
  adding: CHEMBL131890/output5.pdb (deflated 81%)
  adding: CHEMBL131890/output4.pdb (deflated 81%)
  adding: CHEMBL131890/output9.pdb (deflated 81%)
  adding: CHEMBL131890/output1.pdb (deflated 81%)
  adding: CHEMBL131890/output3.pdb (deflated 81%)
  adding: CHEMBL131890/output8.pdb (deflated 81%)
  adding: CHEMBL131890/BiP.pdbqt (deflated 76%)
  adding: CHEMBL131890/output.pdbqt (deflated 86%)
  adding: CHEMBL131890/ligand.pdbqt (deflated 74%)
CHEMBL14249
  adding: CHEMBL14249/ (sto